# Derive BT.709 RGB → CIE XYZ Matrix (SMPTE RP 177 §3.3.1–3.3.7)

This notebook:
- reads **primary chromaticities** and **reference white chromaticity** from **ITU‑R BT.709** (table near the top of page 3 of the Recommendation),
- then follows **SMPTE RP 177** (sections **3.3.1 through 3.3.7**) to derive the **normalized primary matrix (NPM)** that maps *linear* BT.709 RGB to CIE XYZ,
- using **NumPy** for matrix operations.

Notes:
- The procedure assumes **linear** RGB (i.e., after inverse OETF / transfer decoding if starting from non-linear signals).
- RGB is normalized such that **reference white maps from R=G=B=1** and has **Y = 1.0** as required by RP 177.

## Input chromaticities (from ITU‑R BT.709)

CIE 1931 (x, y) chromaticities:

- **Red**: (0.640, 0.330)  
- **Green**: (0.300, 0.600)  
- **Blue**: (0.150, 0.060)  
- **Reference white (D65)**: (0.3127, 0.3290)

We will use these values directly as the input to RP 177 §3.3.1.

In [ ]:
import numpy as np

# ---- ITU-R BT.709 chromaticities (CIE 1931 x,y) ----
xR, yR = 0.640, 0.330
xG, yG = 0.300, 0.600
xB, yB = 0.150, 0.060

# Reference white D65 for BT.709
xW, yW = 0.3127, 0.3290

primaries_xy = np.array([[xR, yR],
                         [xG, yG],
                         [xB, yB]], dtype=np.float64)

white_xy = np.array([xW, yW], dtype=np.float64)

primaries_xy, white_xy

(array([[0.64, 0.33],
        [0.3 , 0.6 ],
        [0.15, 0.06]]),
 array([0.3127, 0.329 ]))

## SMPTE RP 177 §3.3 step-by-step

We follow RP 177 exactly:

- **§3.3.2**: compute z = 1 − (x + y) for each primary and the white.
- **§3.3.3**: form matrix **P** from the (x, y, z) of primaries; form **W** vector from the white, normalized so that **Y = 1.0**.
- **§3.3.4**: compute scaling coefficients **Cᵢ** as `P⁻¹ · W`.
- **§3.3.5**: form diagonal matrix **C** from Cᵢ.
- **§3.3.6**: compute **NPM = P · C**.
- **§3.3.7**: NPM is the RGB→XYZ matrix for linear signals.

In [2]:
def xy_to_xyz_row(x: float, y: float) -> tuple[float, float, float]:
    """RP 177 §3.3.2: given x,y, compute z."""
    z = 1.0 - (x + y)
    return (x, y, z)

# §3.3.2: compute z for primaries and white
xR, yR, zR = xy_to_xyz_row(xR, yR)
xG, yG, zG = xy_to_xyz_row(xG, yG)
xB, yB, zB = xy_to_xyz_row(xB, yB)

xW, yW, zW = xy_to_xyz_row(xW, yW)

(zR, zG, zB, zW)

(0.030000000000000027, 0.10000000000000009, 0.79, 0.35830000000000006)

In [3]:
# §3.3.3: form P matrix and W vector

P = np.array([[xR, xG, xB],
              [yR, yG, yB],
              [zR, zG, zB]], dtype=np.float64)

W = np.array([[xW / yW],
              [1.0],
              [zW / yW]], dtype=np.float64)

P, W

(array([[0.64, 0.3 , 0.15],
        [0.33, 0.6 , 0.06],
        [0.03, 0.1 , 0.79]]),
 array([[0.95045593],
        [1.        ],
        [1.08905775]]))

In [4]:
# §3.3.4: compute scaling coefficients C_i = P^{-1} · W
P_inv = np.linalg.inv(P)
C_vec = P_inv @ W   # shape (3,1): [C_R, C_G, C_B]^T

C_R, C_G, C_B = C_vec.flatten().tolist()
C_R, C_G, C_B

(0.6443606238530613, 1.1919477979462598, 1.2032052560122288)

In [5]:
# §3.3.5: diagonal matrix C from coefficients
C = np.diag([C_R, C_G, C_B])

# §3.3.6: NPM = P · C
NPM = P @ C

NPM

array([[0.4123908 , 0.35758434, 0.18048079],
       [0.21263901, 0.71516868, 0.07219232],
       [0.01933082, 0.11919478, 0.95053215]])

In [6]:
# Display NPM with high precision and with 4-decimal rounding (RP 177 §3.2 output convention)

np.set_printoptions(precision=12, suppress=False)

print("NPM (high precision):")
print(NPM)

print("\nNPM (rounded to 4 decimals):")
print(np.round(NPM, 4))

NPM (high precision):
[[0.412390799266 0.357584339384 0.180480788402]
 [0.212639005872 0.715168678768 0.072192315361]
 [0.019330818716 0.119194779795 0.95053215225 ]]

NPM (rounded to 4 decimals):
[[0.4124 0.3576 0.1805]
 [0.2126 0.7152 0.0722]
 [0.0193 0.1192 0.9505]]


## Sanity checks

1) **White check** (RP 177 normalization): for RGB = [1,1,1]ᵀ, the resulting XYZ should correspond to the white chromaticity with Y=1.

2) **Column meaning**: columns of NPM are the XYZ tristimulus values of the normalized R, G, B primaries (as scaled by Cᵢ).

In [7]:
# 1) White check: XYZ_white = NPM @ [1,1,1]^T should equal [xW/yW, 1, zW/yW]^T
RGB_white = np.array([[1.0],[1.0],[1.0]], dtype=np.float64)
XYZ_from_matrix = NPM @ RGB_white

print("XYZ from NPM @ [1,1,1]:\n", XYZ_from_matrix.flatten())
print("Target W vector:       \n", W.flatten())

print("\nDifference:")
print((XYZ_from_matrix - W).flatten())

XYZ from NPM @ [1,1,1]:
 [0.950455927052 1.             1.08905775076 ]
Target W vector:       
 [0.950455927052 1.             1.08905775076 ]

Difference:
[ 0.00000000000e+00 -2.22044604925e-16  0.00000000000e+00]


In [8]:
# 2) Optional: derive the inverse matrix (XYZ -> RGB) for completeness
NPM_inv = np.linalg.inv(NPM)

print("NPM inverse (XYZ -> RGB), high precision:")
print(NPM_inv)

print("\nNPM inverse rounded to 6 decimals:")
print(np.round(NPM_inv, 6))

NPM inverse (XYZ -> RGB), high precision:
[[ 3.240969941905 -1.53738317757  -0.498610760293]
 [-0.969243636281  1.875967501508  0.041555057407]
 [ 0.055630079697 -0.203976958889  1.056971514243]]

NPM inverse rounded to 6 decimals:
[[ 3.24097  -1.537383 -0.498611]
 [-0.969244  1.875968  0.041555]
 [ 0.05563  -0.203977  1.056972]]


## Result

The **NPM** computed above is the **BT.709 / sRGB D65 linear RGB → CIE XYZ** matrix (within floating-point tolerance),
as obtained directly by the RP 177 procedure using the BT.709 primary and white chromaticities.

## Verification Against Colour Reference Values

This checks the derived BT.709 matrices against Colour's built-in reference for `ITU-R BT.709`.

In [ ]:
from colour.models import RGB_COLOURSPACES

cs_bt709 = RGB_COLOURSPACES["ITU-R BT.709"]
M_ref = np.asarray(cs_bt709.matrix_RGB_to_XYZ, dtype=np.float64)
M_inv_ref = np.asarray(cs_bt709.matrix_XYZ_to_RGB, dtype=np.float64)

delta_M = NPM - M_ref
delta_M_inv = NPM_inv - M_inv_ref

max_abs_M = np.max(np.abs(delta_M))
max_abs_M_inv = np.max(np.abs(delta_M_inv))

print("Colour reference colourspace:", cs_bt709.name)
print("Max abs diff (RGB->XYZ):", max_abs_M)
print("Max abs diff (XYZ->RGB):", max_abs_M_inv)

tolerance = 1e-12
assert np.allclose(NPM, M_ref, atol=tolerance, rtol=0.0), "NPM does not match Colour reference within tolerance."
assert np.allclose(NPM_inv, M_inv_ref, atol=tolerance, rtol=0.0), "NPM_inv does not match Colour reference within tolerance."

print(f"PASS: Both matrices match Colour reference within atol={tolerance}.")